In [ ]:
# TODO: Set your name and task here
annotator_name = None
task = "reannotate" # or "calibrate" or "reannotate"

In [ ]:
import os
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf

from annotate import *


cfg = OmegaConf.load("../../config/default.yaml")

# Resolve paths
project_root = Path("../..").resolve()

# Define paths based on the configuration
hf_data = project_root / cfg.data.paths.hf
output_file = project_root / cfg.data.paths.reannotation_file if task == "reannotate" else project_root / cfg.data.paths.calibration_file

# Load labels into a DataFrame
df = pd.DataFrame()
splits = ['dev_seen', 'dev_unseen', 'test_seen', 'test_unseen', 'train']
for split in splits:
    df_tmp = pd.read_json(f'{hf_data}/{split}.jsonl', lines=True)
    df_tmp['split'] = split
    df = pd.concat([df, df_tmp])
    
# Keep only locally available images and remove duplicates
hf_images = os.listdir(hf_data / "img")
df['image_found'] = df['img'].str.lstrip('img/').isin(hf_images)
df = df[df['image_found'] == True]
df = df.drop_duplicates(subset=['img'], keep='first')

# Keep only images of the relevant split
with open(project_root / cfg.data.paths.base / f"images_to_{task}.txt", "r") as f:
    lines = f.readlines()
    images_to_annotate = [int(line.strip()) for line in lines]
df = df[df['id'].isin(images_to_annotate)]

# Add full path to the image
df['img_path'] = df['img'].apply(lambda x: hf_data / x)

print(f"Total images to {task}: {len(df)}")

In [ ]:
if annotator_name is None:
    raise ValueError("Please set the 'annotator_name' variable with your name before running the annotation.")
else:
    annotate_data(df, output_file, annotator_name)